In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import re

INPUT_CSV = 'csv_input/07290012_modified_shapefile.csv'
OUTPUT_CSV = 'csv_input/shape_id_renamed.csv'

df = pd.read_csv(INPUT_CSV)
# df = df[['County', 'Shape_ID', 'geometry']]

gdf = gpd.GeoDataFrame(df, geometry=gpd.GeoSeries.from_wkt(df['geometry']))
gdf['centroid'] = gdf.geometry.centroid

county_order = df['County'].unique().tolist()

In [ ]:
# Compute each county's centroid as the area-weighted average of its polygon centroids
# (avoids unary_union, which requires topologically valid geometries)
def weighted_county_centroid(group):
    areas = group.geometry.area
    total_area = areas.sum()
    cx = (group['centroid'].x * areas).sum() / total_area
    cy = (group['centroid'].y * areas).sum() / total_area
    return cx, cy

county_centroids = (
    gdf.groupby('County', observed=True)
    .apply(weighted_county_centroid, include_groups=False)
)

def assign_clockwise_ids(group, county_centroid):
    cx, cy = county_centroid

    group = group.copy()
    group['_dx'] = group['centroid'].x - cx
    group['_dy'] = group['centroid'].y - cy
    group['Dist_to_Centroid'] = np.hypot(group['_dx'], group['_dy'])
    group['_angle'] = np.arctan2(group['_dy'], group['_dx'])  # counter-clockwise radians

    # _001 = nearest to county centroid
    nearest_idx = group['Dist_to_Centroid'].idxmin()
    start_angle = group.loc[nearest_idx, '_angle']

    # Clockwise: negate angles; shift so start_angle = 0; wrap to [0, 2pi)
    group['_cw_angle'] = (-(group['_angle'] - start_angle)) % (2 * np.pi)

    # Nearest polygon gets 0 -> sorts first
    group = group.sort_values('_cw_angle').reset_index(drop=True)

    # Assign Shape_ID
    county = group['County'].iloc[0]
    group['Shape_ID'] = [f"{county}_{i+1:03d}" for i in range(len(group))]

    return group.drop(columns=['_dx', '_dy', '_angle', '_cw_angle'])

gdf['County'] = pd.Categorical(gdf['County'], categories=county_order, ordered=True)

result_parts = []
for county in county_order:
    group = gdf[gdf['County'] == county]
    if group.empty:
        continue
    numbered = assign_clockwise_ids(group, county_centroids[county])
    result_parts.append(numbered)

gdf = pd.concat(result_parts, ignore_index=True)

gdf[['County', 'Shape_ID', 'Dist_to_Centroid']]

In [ ]:
print("=== Shape_ID Validation ===\n")
all_valid = True

for county in county_order:
    group = gdf[gdf['County'] == county]
    ids = group['Shape_ID'].tolist()
    n = len(ids)
    expected = [f"{county}_{i:03d}" for i in range(1, n + 1)]

    bad_format = [s for s in ids if not re.fullmatch(rf"{re.escape(county)}_\d{{3}}", s)]
    missing = set(expected) - set(ids)
    duplicates = [s for s in ids if ids.count(s) > 1]

    if bad_format or missing or duplicates:
        all_valid = False
        print(f"[FAIL] {county} ({n} polygons)")
        if bad_format:   print(f"  Bad format:  {bad_format}")
        if missing:      print(f"  Missing IDs: {sorted(missing)}")
        if duplicates:   print(f"  Duplicates:  {list(set(duplicates))}")
    else:
        print(f"[ OK ] {county} \u2014 {county}_001 to {county}_{n:03d}")

print("\nAll Shape_IDs valid." if all_valid else "\nIssues found \u2014 review above.")

In [ ]:
gdf.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {len(gdf)} rows to {OUTPUT_CSV}")